In [ ]:
!git clone https://github.com/B-I-T-W-I-S-E-M-I-N-D-S/mimi-to-hubert-bridge-v3.git

Cloning into 'mimi-to-hubert-bridge'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 31 (delta 12), reused 23 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 45.86 KiB | 838.00 KiB/s, done.
Resolving deltas: 100% (12/12), done.


In [4]:
!pip install -q moshi safetensors huggingface_hub onnxruntime-gpu librosa tqdm


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [7]:
# !apt update && apt install -y unzip
!apt-get update -qq && apt-get install -y aria2 unzip

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
aria2 is already the newest version (1.36.0-1).
Suggested packages:
  zip
The following NEW packages will be installed:
  unzip
0 upgraded, 1 newly installed, 0 to remove and 146 not upgraded.
Need to get 175 kB of archives.
After this operation, 386 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 unzip amd64 6.0-26ubuntu3.2 [175 kB]
Fetched 175 kB in 0s (937 kB/s)
debconf: delaying package configuration, since apt-utils is not installed
Selecting previously unselected package unzip.
(Reading database ... 24220 files and directories currently installed.)
Preparing to unpack .../unzip_6.0-26ubuntu3.2_amd64.deb ...
Unpacking unzip (6.0-26ubuntu3.2) ...
Setting up unzip (6.0-26ubuntu3.2) ...
Processing triggers for mailcap (3.70+nmu1ubuntu1) ...


In [ ]:
from huggingface_hub import snapshot_download
import os


repo_id = "Darknsu/mimi-hubert-librispeech-10val-data-preprocess"  # make sure this is actually a dataset repo
download_dir = "/workspace/datahf"

# Create directory if it doesn't exist
os.makedirs(download_dir, exist_ok=True)

print("📥 Downloading entire dataset...")

# Download full dataset repo
snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",   # ✅ changed from "model" to "dataset"
    local_dir=download_dir,
    local_dir_use_symlinks=False
)

print("✅ All files downloaded to:", download_dir)

📥 Downloading entire dataset...


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

data.zip:   0%|          | 0.00/34.5G [00:00<?, ?B/s]

✅ All files downloaded to: /workspace/datahf


In [24]:
!apt-get install -y p7zip-full pigz
!7z x -mmt=on /workspace/datahf/data.zip -o/workspace/datahf/ -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  p7zip
Suggested packages:
  p7zip-rar
The following NEW packages will be installed:
  p7zip p7zip-full pigz
0 upgraded, 3 newly installed, 0 to remove and 146 not upgraded.
Need to get 1612 kB of archives.
After this operation, 6009 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 pigz amd64 2.6-1 [63.6 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 p7zip amd64 16.02+dfsg-8 [363 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 p7zip-full amd64 16.02+dfsg-8 [1186 kB]
Fetched 1612 kB in 0s (4290 kB/s)   
debconf: delaying package configuration, since apt-utils is not installed
Selecting previously unselected package pigz.
(Reading database ... 24238 files and directories currently installed.)
Preparing to unpack .../archives/pigz_2.6-1_amd64.deb ...
Unpac

In [8]:
!unzip /workspace/data.zip

Archive:  /workspace/data.zip
   creating: content/mimi-to-hubert-bridge/data/
   creating: content/mimi-to-hubert-bridge/data/LibriSpeech/
  inflating: content/mimi-to-hubert-bridge/data/LibriSpeech/LICENSE.TXT  
  inflating: content/mimi-to-hubert-bridge/data/LibriSpeech/SPEAKERS.TXT  
  inflating: content/mimi-to-hubert-bridge/data/LibriSpeech/BOOKS.TXT  
   creating: content/mimi-to-hubert-bridge/data/LibriSpeech/train-clean-100/
   creating: content/mimi-to-hubert-bridge/data/LibriSpeech/train-clean-100/103/
   creating: content/mimi-to-hubert-bridge/data/LibriSpeech/train-clean-100/103/1241/
  inflating: content/mimi-to-hubert-bridge/data/LibriSpeech/train-clean-100/103/1241/103-1241-0010.flac  
  inflating: content/mimi-to-hubert-bridge/data/LibriSpeech/train-clean-100/103/1241/103-1241-0011.flac  
  inflating: content/mimi-to-hubert-bridge/data/LibriSpeech/train-clean-100/103/1241/103-1241-0019.flac  
  inflating: content/mimi-to-hubert-bridge/data/LibriSpeech/train-clean-100/1

In [9]:
import os
from huggingface_hub import hf_hub_download

# Define target directory
base_dir = "/workspace/mimi-to-hubert-bridge"
model_dir = os.path.join(base_dir, "model")

# Create directories if they don't exist
os.makedirs(model_dir, exist_ok=True)

# Download the file into the model folder
file_path = hf_hub_download(
    repo_id="digital-avatar/ditto-talkinghead",
    filename="ditto_pytorch/aux_models/hubert_streaming_fix_kv.onnx",
    local_dir=model_dir,
    local_dir_use_symlinks=False
)

print("Downloaded to:", file_path)

ditto_pytorch/aux_models/hubert_streamin(…):   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Downloaded to: /workspace/mimi-to-hubert-bridge/model/ditto_pytorch/aux_models/hubert_streaming_fix_kv.onnx


In [10]:
import shutil
import os

src = "/workspace/mimi-to-hubert-bridge/model/ditto_pytorch/aux_models/hubert_streaming_fix_kv.onnx"
dst = "/workspace/mimi-to-hubert-bridge/model/hubert_streaming_fix_kv.onnx"

# Check if source file exists
if os.path.exists(src):
    shutil.move(src, dst)
    print("✅ File moved successfully!")
else:
    print("❌ Source file not found!")

✅ File moved successfully!


In [ ]:
%cd mimi-to-hubert-bridge-v3

/workspace/mimi-to-hubert-bridge


/usr/local/lib/python3.11/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [12]:
!pip install -r requirements.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached huggingface_hub-1.8.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 62.6 MB/s eta 0:00:00
  Created wheel for pyworld: filename=pyworld-0.3.5-cp311-cp311-linux_x86_64.whl size=910168 sha256=fcdf384511b78483ec9d5ce6eda5d3962ac8650ca0d8b7d934106ae715460af3
  Stored in directory: /root/.cache/pip/wheels/26/f0/db/ebcd5cdfe5ad7d229917d3a8

In [25]:
!mkdir -p /workspace/mimi-to-hubert-bridge/data/LibriSpeech

# Download with progress bar
!aria2c -x 16 -s 16 https://www.openslr.org/resources/12/train-clean-100.tar.gz

# Extract quietly (no file listing)
!tar -xf train-clean-100.tar.gz

# Move extracted folder
!mv LibriSpeech/train-clean-100 /workspace/mimi-to-hubert-bridge/data/LibriSpeech

# Clean up
!rm -rf train-clean-100.tar.gz LibriSpeech

print("✅ Done! Data is in /workspace/mimi-to-hubert-bridge/data/")


03/31 06:51:57 [NOTICE] Downloading 1 item(s)
 *** Download Progress Summary as of Tue Mar 31 06:52:58 2026 ***              1m2s]mmm
[#56b6cb 2.1GiB/5.9GiB(35%) CN:16 DL:64MiB ETA:1m]
FILE: /workspace/mimi-to-hubert-bridge/train-clean-100.tar.gz
-------------------------------------------------------------------------------

[#56b6cb 5.9GiB/5.9GiB(99%) CN:5 DL:71MiB]]m
03/31 06:53:53 [NOTICE] Download complete: /workspace/mimi-to-hubert-bridge/train-clean-100.tar.gz

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
56b6cb|OK  |    52MiB/s|/workspace/mimi-to-hubert-bridge/train-clean-100.tar.gz

Status Legend:
(OK):download completed.
✅ Done! Data is in /workspace/mimi-to-hubert-bridge/data/


In [26]:
!ls ./data/LibriSpeech/train-clean-100

103   1502  2002  2518	3168  3807  426   4853	5750  6437  7226  8063	8630
1034  1553  2007  254	32    3830  4267  4859	5778  6454  7264  8088	87
1040  1578  201   26	3214  3857  4297  4898	5789  6476  7278  8095	8747
1069  1594  2092  2691	322   3879  4340  5022	5808  6529  730   8098	8770
1081  1624  211   27	3235  39    4362  5049	5867  6531  7302  8108	8797
1088  163   2136  2764	3240  3947  4397  5104	587   6563  7312  8123	8838
1098  1723  2159  2817	3242  3982  4406  5163	60    669   7367  8226	887
1116  1737  2182  2836	3259  3983  441   5192	6000  6818  7402  8238	89
118   1743  2196  2843	328   40    4441  5322	6019  6836  7447  83	8975
1183  1841  226   289	332   4014  445   5339	6064  6848  7505  831	909
1235  1867  2289  2893	3374  4018  446   5390	6078  6880  7511  8312	911
1246  1898  229   2910	3436  403   4481  5393	6081  6925  7517  8324
125   19    233   2911	3440  405   458   5456	6147  696   7635  839
1263  1926  2384  2952	3486  4051  460   5463	6181  7059  7780  8

In [21]:
# !python preprocess.py \
#     --dataset librispeech \
#     --root ./data/LibriSpeech/train-clean-100 \
#     --out_dir data \
#     --val_frac 0.01 \
#     --preextract \
#     --device cuda

In [22]:
# !python preprocess.py \
#     --dataset librispeech \
#     --root ./data/LibriSpeech/train-clean-100 \
#     --out_dir data \
#     --val_frac 0.01 \
#     --preextract \
#     --device cuda \
#     --num_workers 16   # tune up if your disk is fast (8–16 for NVMe)

In [14]:
!torchrun --nproc_per_node=5 preprocess.py \
    --dataset librispeech \
    --root ./data/LibriSpeech/train-clean-100 \
    --out_dir data \
    --val_frac 0.1 \
    --preextract \
    --device cuda \
    --num_workers 16

W0331 06:20:28.372000 138813522661376 torch/distributed/run.py:779] 
W0331 06:20:28.372000 138813522661376 torch/distributed/run.py:779] *****************************************
W0331 06:20:28.372000 138813522661376 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0331 06:20:28.372000 138813522661376 torch/distributed/run.py:779] *****************************************
[rank 0] 2026-03-31 06:20:31,037 INFO Running with 5 process(es) / GPU(s)
[rank 0] 2026-03-31 06:20:31,048 INFO Discovering librispeech audio under ./data/LibriSpeech/train-clean-100
[rank 0] 2026-03-31 06:20:31,049 INFO Found 20 LibriSpeech utterances
[rank 0] 2026-03-31 06:20:31,049 INFO Wrote 18 records → data/train.jsonl
[rank 0] 2026-03-31 06:20:31,050 INFO Wrote 2 records → data/val.jsonl
[rank 0] 2026-03-31 06:20:31,050 IN

In [36]:
!rm -rf /workspace/mimi-to-hubert-bridge/checkpoints

In [15]:
# !python train.py --config config.yaml

In [ ]:
!torchrun --nproc_per_node=5 train.py --config config.yaml

W0331 08:43:35.519000 134236497100800 torch/distributed/run.py:779] 
W0331 08:43:35.519000 134236497100800 torch/distributed/run.py:779] *****************************************
W0331 08:43:35.519000 134236497100800 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0331 08:43:35.519000 134236497100800 torch/distributed/run.py:779] *****************************************
2026-03-31 08:43:38,525  INFO      [rank57966]  World size: 5 GPU(s) | Device: cuda:0 | AMP: True
2026-03-31 08:43:38,525  INFO      [rank57966]    GPU 0: NVIDIA RTX A4000
2026-03-31 08:43:38,526  INFO      [rank57966]    GPU 1: NVIDIA RTX A4000
2026-03-31 08:43:38,526  INFO      [rank57966]    GPU 2: NVIDIA RTX A4000
2026-03-31 08:43:38,526  INFO      [rank57966]    GPU 3: NVIDIA RTX A4000
2026-03-31 08:43:38,526  INFO      [ran

In [27]:
!pip uninstall -y huggingface_hub
!pip install --upgrade --no-cache-dir huggingface_hub

Found existing installation: huggingface_hub 1.8.0
Uninstalling huggingface_hub-1.8.0:
  Successfully uninstalled huggingface_hub-1.8.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 30.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
moshi 0.2.13 requires huggingface-hub<1.0.0,>=0.24, but you have huggingface-hub 1.8.0 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
# # !pip install -q huggingface_hub
# from huggingface_hub import login

# # Paste your token here
# hf_token = ""
# login(token=hf_token)



# import os
# from huggingface_hub import HfApi
# from tqdm import tqdm

# # --- CONFIGURATION ---

# # Local folder you want to upload
# local_folder = "/workspace/mimi-to-hubert-bridge/upload"  # Replace with your folder path

# # Hugging Face dataset repo info
# repo_id = "Darknsu/mimi-hubert-librispeech-10val-data-preprocess"  # Replace with your repo
# repo_type = "dataset"

# # Commit message
# commit_message = "Upload entire folder with structure in one commit"

# # --- UPLOAD FUNCTION ---
# def upload_folder_single_commit(local_folder, repo_id, repo_type="dataset"):
#     api = HfApi()

#     print("📤 Uploading folder to Hugging Face with a single commit...")

#     # Progress bar for local file count (optional)
#     total_files = sum(len(files) for _, _, files in os.walk(local_folder))
#     with tqdm(total=total_files, desc="Processing files") as pbar:
#         for _, _, files in os.walk(local_folder):
#             pbar.update(len(files))

#     # Upload folder in a single commit
#     api.upload_folder(
#         folder_path=local_folder,
#         repo_id=repo_id,
#         repo_type=repo_type,
#         commit_message=commit_message,
#     )

#     print("✅ Folder uploaded successfully in a single commit.")

# # --- EXECUTE ---
# upload_folder_single_commit(local_folder, repo_id, repo_type)

📤 Uploading folder to Hugging Face with a single commit...


Processing files: 100%|██████████| 1/1 [00:00<00:00, 7084.97it/s]


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

In [3]:
!apt update && apt install -y zip

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1581 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:3 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2475 kB]
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease              
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease              
Fetched 2477 kB in 1s (3552 kB/s)m                  
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
145 packages can be upgraded. Run 'apt list --upgradable' to see them.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  unzip
The following NEW packages will be ins

In [5]:
!zip -r -q /workspace/mimi-to-hubert-bridge/data.zip /workspace/mimi-to-hubert-bridge/data

In [ ]:
import os
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="https://s3api-eu-ro-1.runpod.io",  # replace with your endpoint
    aws_access_key_id="YOUR_KEY",
    aws_secret_access_key="YOUR_SECRET"
)

def upload_folder(local_folder, bucket, s3_prefix=""):
    for root, dirs, files in os.walk(local_folder):
        for file in files:
            local_path = os.path.join(root, file)

            # keep folder structure in S3
            relative_path = os.path.relpath(local_path, local_folder)
            s3_path = os.path.join(s3_prefix, relative_path).replace("\\", "/")

            print(f"Uploading {local_path} → {s3_path}")
            s3.upload_file(local_path, bucket, s3_path)

# usage
upload_folder("/workspace/mimi-to-hubert-bridge/data", "7nreujc6f2", "backup")